# Itô vs Stratonovich

Chapter 19: Stochastic Differential Equations · *Mathematics for AI and Machine Learning* by Fuheng Wu

One-step schematic: left endpoint vs midpoint stochastic integral.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-19/ito-stratonovich/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.manual_seed(42)

# ∫_0^T W dW: the answer depends on where in each step you evaluate W
T, n_steps, n_paths = 1.0, 2000, 20000
dt = T / n_steps
dW = dt ** 0.5 * torch.randn(n_paths, n_steps, dtype=torch.float64)
W = torch.cat([torch.zeros(n_paths, 1, dtype=torch.float64), dW.cumsum(1)], 1)
left, right = W[:, :-1], W[:, 1:]

ito = (left * dW).sum(1)                       # left endpoint (Ito)
strat = (0.5 * (left + right) * dW).sum(1)     # midpoint average (Stratonovich)
WT = W[:, -1]

qv = (dW ** 2).sum(1)                           # quadratic variation, -> T as dt -> 0
print("Ito sum          == (W_T^2 - sum dW^2)/2 exactly: max diff", (ito - (WT ** 2 - qv) / 2).abs().max().item())
print("Stratonovich sum == W_T^2 / 2 exactly           : max diff", (strat - WT ** 2 / 2).abs().max().item())
print(f"sum dW^2 over paths: mean {qv.mean().item():.4f}, std {qv.std().item():.4f} (-> T = {T} with no spread as dt -> 0)")
print("so in the limit the Ito integral is (W_T^2 - T)/2")
print(f"\nE[Ito integral] = {ito.mean().item():+.4f} (0: it is a martingale)")
print(f"E[Stratonovich] = {strat.mean().item():+.4f} (T/2 = {T / 2})")
print(f"difference Stratonovich - Ito = {(strat - ito).mean().item():.4f}  = half the quadratic variation T/2")

# The gap is exactly (1/2) sum dW^2, which tends to T/2: it comes from dW^2 ~ dt
print("(1/2) sum dW^2 on one path =", 0.5 * (dW[0] ** 2).sum().item())

Ito sum          == (W_T^2 - sum dW^2)/2 exactly: max diff 3.4638958368304884e-14
Stratonovich sum == W_T^2 / 2 exactly           : max diff 3.419486915845482e-14
sum dW^2 over paths: mean 1.0000, std 0.0316 (-> T = 1.0 with no spread as dt -> 0)
so in the limit the Ito integral is (W_T^2 - T)/2

E[Ito integral] = -0.0002 (0: it is a martingale)
E[Stratonovich] = +0.4998 (T/2 = 0.5)
difference Stratonovich - Ito = 0.5000  = half the quadratic variation T/2
(1/2) sum dW^2 on one path = 0.5103406502036993
